# 06 — Pitch Opportunity Ranker

**Purpose:** Automatically rank which clients to pitch next based on growth potential, churn risk, and market position.  
**Key question:** Where should the sales team focus their energy this quarter?  

**Tables used:** `marts.mart_pitch_opportunities`

In [ ]:
from google.cloud import bigquery
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import os
PROJECT = os.environ.get('BQ_PROJECT', 'fmn-sandbox')
LOCATION = 'africa-south1'
client = bigquery.Client(project=PROJECT, location=LOCATION)

def q(sql):
    return client.query(sql).to_dataframe()

print(f'Connected to {PROJECT} ({LOCATION})')

## 1. Top 20 pitch opportunities
Ranked by composite pitch score (market size + growth gap + low churn + spend efficiency).

In [ ]:
opps = q(f"""
    SELECT CATEGORY_TWO, DESTINATION, customers,
           ROUND(total_spend, 0) AS total_spend,
           market_share_pct, penetration_pct,
           ROUND(addressable_market, 0) AS addressable_market,
           gap_to_leader_pct,
           ROUND(avg_churn_probability * 100, 1) AS churn_risk_pct,
           pitch_score,
           recommended_action
    FROM `{PROJECT}.marts.mart_pitch_opportunities`
    ORDER BY pitch_score DESC
    LIMIT 20
""")

if not opps.empty:
    print(f'Top {len(opps)} pitch opportunities:')
    display(opps)
else:
    print('No data. Run mart_pitch_opportunities SQL first.')

## 2. Opportunity map
X = market share (where they are now), Y = addressable market (how much room to grow). Size = customers.

In [ ]:
if not opps.empty:
    fig = px.scatter(opps, x='market_share_pct', y='addressable_market',
                     size='customers', color='recommended_action',
                     hover_name='DESTINATION',
                     title='Pitch opportunity map: market share vs addressable market',
                     labels={'market_share_pct': 'Current market share (%)',
                             'addressable_market': 'Addressable market (R)'})
    fig.update_layout(height=600)
    fig.show()

## 3. Breakdown by recommended action
How many clients fall into each action bucket?

In [ ]:
actions = q(f"""
    SELECT recommended_action,
           COUNT(*) AS num_clients,
           ROUND(SUM(total_spend), 0) AS total_spend,
           ROUND(SUM(addressable_market), 0) AS total_addressable,
           ROUND(AVG(pitch_score), 2) AS avg_pitch_score
    FROM `{PROJECT}.marts.mart_pitch_opportunities`
    GROUP BY recommended_action
    ORDER BY avg_pitch_score DESC
""")

if not actions.empty:
    fig = px.bar(actions, x='recommended_action', y='num_clients',
                 color='total_addressable', color_continuous_scale='Greens',
                 title='Clients by recommended action',
                 labels={'num_clients': 'Number of clients', 'total_addressable': 'Addressable market (R)'})
    fig.show()
    display(actions)

## 4. Top opportunities by category
For each category, who is the best client to pitch?

In [ ]:
top_per_cat = q(f"""
    WITH ranked AS (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY CATEGORY_TWO ORDER BY pitch_score DESC) AS rn
        FROM `{PROJECT}.marts.mart_pitch_opportunities`
        WHERE spend_rank > 1
    )
    SELECT CATEGORY_TWO, DESTINATION,
           ROUND(total_spend, 0) AS total_spend,
           market_share_pct, gap_to_leader_pct,
           ROUND(addressable_market, 0) AS addressable_market,
           pitch_score, recommended_action
    FROM ranked
    WHERE rn = 1
    ORDER BY addressable_market DESC
    LIMIT 20
""")

if not top_per_cat.empty:
    print('Best pitch target per category (excluding leaders):')
    display(top_per_cat)
else:
    print('No non-leader destinations found.')

## 5. High-value defend list
These are market leaders with high churn risk — they need retention campaigns.

In [ ]:
defend = q(f"""
    SELECT CATEGORY_TWO, DESTINATION, customers,
           ROUND(total_spend, 0) AS total_spend,
           market_share_pct,
           ROUND(avg_churn_probability * 100, 1) AS churn_risk_pct,
           high_risk_customers,
           recommended_action
    FROM `{PROJECT}.marts.mart_pitch_opportunities`
    WHERE spend_rank <= 3
      AND avg_churn_probability > 0.2
    ORDER BY high_risk_customers DESC
    LIMIT 15
""")

if not defend.empty:
    print('Market leaders with high churn exposure:')
    display(defend)
else:
    print('No leaders with significant churn risk found. Thats a good sign.')